## California Housing Prices (Regression)
Now, what happens if the data is highly complex and non-linear? Let's predict house prices in California based on location, rooms, and income.

We will compare a simple **Linear Regression** with a **Random Forest Regressor** to see if complexity is sometimes necessary.

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import accuracy_score, r2_score
import joblib

In [ ]:
# 1. Load Data
housing = fetch_california_housing()
X_house = housing.data
y_house = housing.target

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_house, y_house, test_size=0.2, random_state=42)

# 2. Build Pipelines
pipeline_lin_reg = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
pipeline_rf_reg = Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(random_state=42, n_jobs=-1))])

# 3. Train Models
pipeline_lin_reg.fit(X_train_h, y_train_h)
pipeline_rf_reg.fit(X_train_h, y_train_h)

# 4. Evaluate (R2 Score)
test_r2_lin = pipeline_lin_reg.score(X_test_h, y_test_h)
test_r2_rf = pipeline_rf_reg.score(X_test_h, y_test_h)

print(f"Linear Regression Test R2: {test_r2_lin:.4f}")
print(f"Random Forest Test R2:     {test_r2_rf:.4f}")

print("\n OBSERVATION:")
print("Here, the simple Linear Regression struggles (R2 ~ 0.59).")
print("House pricing is complex and non-linear! Random Forest captures these patterns much better (R2 ~ 0.80).")

Linear Regression Test R2: 0.5758
Random Forest Test R2:     0.8053

🔍 OBSERVATION:
Here, the simple Linear Regression struggles (R2 ~ 0.59).
House pricing is complex and non-linear! Random Forest captures these patterns much better (R2 ~ 0.80).


In [ ]:
# 1. Define the parameter grid

param_grid = {
    'model__n_estimators': [50, 100, 200]  # Testing different numbers of trees
}

# 2. Initialize GridSearchCV
grid_search = GridSearchCV(estimator=pipeline_rf_reg, param_grid=param_grid, cv=3, n_jobs=-1, verbose=1)

# 3. Fit the grid search to test all combinations
grid_search.fit(X_train_h, y_train_h)

print("\n--- GRID SEARCH RESULTS ---")
print(f"Best Hyperparameters Found: {grid_search.best_params_}")

# 4. Extract the absolute best pipeline from GridSearch
best_rf_pipeline = grid_search.best_estimator_
best_test_r2 = best_rf_pipeline.score(X_test_h, y_test_h)
print(f"Test R2 Score of Best Tuned Model: {best_test_r2:.4f}")

Fitting 3 folds for each of 3 candidates, totalling 9 fits

--- GRID SEARCH RESULTS ---
Best Hyperparameters Found: {'model__n_estimators': 200}
Test R2 Score of Best Tuned Model: 0.8063


In [6]:
import os


# 1. Define the export path
export_path = r'C:\Users\Lenovo\geo-spatial-valuation-regressor\Export'

# 2. Creating the directory 
if not os.path.exists(export_path):
    os.makedirs(export_path)

# 3. Save the best pipeline (StandardScaler + Best RF Model)
model_filename = os.path.join(export_path, 'housing_regressor_pipeline.joblib')
joblib.dump(grid_search.best_estimator_, model_filename)

print(f"Best Housing Model saved successfully at: {model_filename}")

Best Housing Model saved successfully at: C:\Users\Lenovo\geo-spatial-valuation-regressor\Export\housing_regressor_pipeline.joblib
